# 02 - Load the tennis data into BigQuery
**What it does:** <in your own words, one or two sentences>
**Needs:** notebooks 00 and 01 have run
**Produces:** the table tennis_raw.wta_matches (unedited, every column STRING)
**Run after:** 01_ingest_tennis

## Install the pinned libraries

In [ ]:
# The repo is public, so requirements.txt can be read straight from GitHub.
REQUIREMENTS_URL = "https://raw.githubusercontent.com/katiedevane/tennis-project/main/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

## Sign in and load settings from Colab Secrets
Switch on Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"        # the bucket and the BigQuery datasets must share this location
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Find the commit and checksums the ingest notebook recorded

In [ ]:
import json
import pandas as pd
from google.cloud import bigquery, storage

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET_NAME)
bq = bigquery.Client(project=PROJECT_ID, location=REGION)

names = sorted(b.name for b in gcs.list_blobs(BUCKET_NAME, prefix="manifests/wta_matches_"))
m = json.loads(bucket.blob(names[-1]).download_as_text())      # the latest manifest

COMMIT = m["commit"]
RAW_PREFIX = f"raw/tennis_sackmann_archive/{COMMIT[:8]}/wta"
print("manifests found:", len(names), "| using the latest | commit:", COMMIT[:8], "| files:", len(m["files"]))

## Check that every file has the same columns in the same order
BigQuery matches columns by position, so a different header in one year would silently scramble that year.

In [ ]:
headers = {}
for blob in gcs.list_blobs(BUCKET_NAME, prefix=f"{RAW_PREFIX}/wta_matches_20"):
    first_line = blob.download_as_bytes(start=0, end=4095).decode("utf-8-sig").splitlines()[0]
    headers[blob.name.split("/")[-1]] = first_line

distinct = set(headers.values())
print(len(headers), "files,", len(distinct), "distinct header row(s)")
assert len(headers) == len(m["files"]), "File count differs from the manifest"
assert len(distinct) == 1, "Headers differ between files: stop and investigate"

COLUMNS = next(iter(distinct)).split(",")
print(len(COLUMNS), "columns:", COLUMNS)

## Load into BigQuery
Every column is loaded as text, so nothing is guessed at this stage. Re-running replaces the table, so duplicates cannot build up.

In [ ]:
TABLE = f"{PROJECT_ID}.tennis_raw.wta_matches"
uri = f"gs://{BUCKET_NAME}/{RAW_PREFIX}/wta_matches_20*.csv"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    schema=[bigquery.SchemaField(c, "STRING") for c in COLUMNS],
    write_disposition="WRITE_TRUNCATE",
    max_bad_records=0,          # any malformed row makes the job fail loudly
)
job = bq.load_table_from_uri(uri, TABLE, job_config=job_config)
job.result()
print("Load job:", job.job_id)
print("Rows loaded:", job.output_rows)

## Check: does BigQuery hold exactly what the download recorded?
The rows of each source file are counted by BigQuery (through an external table over the same files) and compared with the row counts in the manifest. The total loaded into the table must equal the manifest total.

Counting by the year in `tourney_date` would not work as a check: tournaments that start in the last days of December are stored in the next season's file, so their rows fall in a different year than their file.

In [ ]:
ext = bigquery.ExternalConfig("CSV")
ext.source_uris = [uri]
ext.schema = [bigquery.SchemaField(c, "STRING") for c in COLUMNS]
ext.options.skip_leading_rows = 1
cfg = bigquery.QueryJobConfig(table_definitions={"src": ext}, maximum_bytes_billed=1_000_000_000)
per_file = {r.f.split("/")[-1]: r.n for r in
            bq.query("SELECT _FILE_NAME AS f, COUNT(*) AS n FROM src GROUP BY f", job_config=cfg).result()}

expected = {e["file"]: e["rows"] for e in m["files"]}
check = pd.DataFrame({"manifest_rows": expected, "bigquery_file_rows": per_file})
check["match"] = check["manifest_rows"] == check["bigquery_file_rows"]
print(check)

loaded_total = bq.get_table(TABLE).num_rows
print("manifest total:", check["manifest_rows"].sum(), "| rows loaded into the table:", loaded_total)
assert check["match"].all(), "A file's row count differs between the manifest and BigQuery"
assert loaded_total == check["manifest_rows"].sum(), "The table does not hold every row of the files"